In [ ]:

from types import SimpleNamespace
import numpy as np
import math

class ED1F:

    driveUnit = SimpleNamespace(**{ 
        'gearBoxGearRatio': 1000.0,        
        'timingBeltTransmissionGearRatio': 2.0,
        'spindlePitch': 5.0,
        'motorIncrementPositions': 8_388_608, 
        'cylinderDiameter': 15.0,
        'limit': {
            'low': 0,
            'high': 838_633_324 
        }
    })

    def gearRatio(self):
        return self.driveUnit.timingBeltTransmissionGearRatio * self.driveUnit.gearBoxGearRatio

    def cylinderArea(self):
        return np.pow(self.driveUnit.cylinderDiameter,2) * np.pi / 4.
    
    def pitchVolume(self):
        return self.driveUnit.spindlePitch * self.cylinderArea()

    def mulmin2incs(self, value):
        # mm/r
        transmission = ED1F.driveUnit.spindlePitch / self.gearRatio()
        # µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        # µl/min
        return np.clip(value / (injectionRateIncrement * 60), self.driveUnit.limit['low'], self.driveUnit.limit['high'])
    
    def incs2mulmin(self, value):
        # mm/r
        transmission = self.driveUnit.spindlePitch / self.gearRatio()
        # mm³/r ~ µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        return (value * injectionRateIncrement * 60.0)
    
    def mms2mulmin(self, value):
        # mm³/s ~ µl/s
        return value * self.cylinderArea() * 60
    
    def turn2ml(self, value):

        bitRange = 2 ** self.encoderUnit.bits[1] -1
        
        mtb = value[0] * self.pitchVolume() / self.gearRatio()
        stb = value[1] / bitRange * self.pitchVolume() / self.gearRatio()
        
        return (mtb + stb) / 1000
        
    def value(self, value, range=32):
        rc = (2**range - 1) + value if value < 0 else value
        return rc
    
    def split(self, value, bits, range=32):            
        value = bin(value)[2:].zfill(range)
        return [
            int(value[:bits].zfill(range),2), 
            int(value[bits:].zfill(range),2)
            ]
    
    def merge(self, value, bits, range=32, verbose=False):          
        rc = self.value(int("".join([
            bin(value[0])[2:].zfill(bits), 
            bin(value[1])[2:].zfill(range-bits)]), 2), range)
        return rc

INCS_MAX = 838_633_324

ed = ED1F()
ed.incs2mulmin(INCS_MAX)


np.float64(2649.9999993599095)